# Práctica 1. El componente probabilístico en la mesa de soporte

Starter de la práctica 1. Instrucciones completas y rúbrica en `practice.md`; el reporte se llena en `report-template.md`.

Cómo usarlo:

* Corre las celdas en orden. Todas corren aunque no hayas llenado los `TODO`; las tablas salen vacías hasta que los llenes.
* Cada `TODO` está marcado. Las predicciones se escriben **antes** de ejecutar la celda que mide, y se conservan aunque fallen.
* Al final, guarda el notebook con las salidas (Archivo → Guardar) y descárgalo como `practica-01-starter.ipynb`.

Costo: cero. GPT-2 corre en la CPU de Colab.

In [ ]:
# En Colab estas librerías ya vienen instaladas; la línea no hace daño si ya están.
!pip install -q transformers torch pandas tabulate

In [ ]:
import re, time, statistics
from collections import Counter
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(0)
tok = AutoTokenizer.from_pretrained("gpt2")
modelo = AutoModelForCausalLM.from_pretrained("gpt2", dtype=torch.float32)
modelo.eval()

def generar(prompt, do_sample=False, temperature=1.0, max_new_tokens=12):
    """Devuelve (texto_generado, latencia_en_segundos). Solo el texto nuevo, sin el prompt."""
    ids = tok(prompt, return_tensors="pt").input_ids
    t0 = time.perf_counter()
    with torch.no_grad():
        out = modelo.generate(ids, max_new_tokens=max_new_tokens, do_sample=do_sample,
                              temperature=temperature if do_sample else None,
                              pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True), time.perf_counter() - t0

print("GPT-2 listo:", round(sum(p.numel() for p in modelo.parameters()) / 1e6), "millones de parámetros")

## Sección 1. Diez requisitos de la mesa de soporte

Para cada requisito decide `"codigo"`, `"llm"` o `"falta_info"` y escribe una línea de justificación. Cuando elijas `falta_info`, di qué dato falta.

In [ ]:
REQUISITOS = {
    1: "Detectar si el ticket contiene un correo electrónico o un teléfono.",
    2: "Clasificar el ticket en una de cinco categorías (facturación, acceso, error técnico, solicitud de función, otro).",
    3: "Marcar prioridad alta si el ticket dice que el sistema está caído para toda una empresa.",
    4: "Calcular el monto a reembolsar cuando se cobró dos veces la mensualidad.",
    5: "Redactar un borrador de respuesta al cliente.",
    6: "Detectar tickets duplicados exactos (el mismo texto enviado dos veces).",
    7: "Resumir un hilo de veinte mensajes para el supervisor.",
    8: "Traducir al español un ticket escrito en inglés.",
    9: "Verificar que el RFC que envía el cliente tiene formato válido.",
    10: "Decidir si un ticket debe escalarse a un humano.",
}

# TODO: llena las diez decisiones ("codigo", "llm" o "falta_info") y las justificaciones.
MI_CLASIFICACION = {i: None for i in REQUISITOS}
MI_JUSTIFICACION = {i: "" for i in REQUISITOS}

VALIDAS = {"codigo", "llm", "falta_info", None}
assert all(v in VALIDAS for v in MI_CLASIFICACION.values()), "usa solo codigo, llm o falta_info"

tabla_requisitos = pd.DataFrame({
    "#": list(REQUISITOS),
    "Requisito": list(REQUISITOS.values()),
    "Decisión": [MI_CLASIFICACION[i] or "" for i in REQUISITOS],
    "Justificación": [MI_JUSTIFICACION[i] for i in REQUISITOS],
})
print("Tabla de requisitos (cópiala al reporte):")
print(tabla_requisitos.to_markdown(index=False))
print("\nDecisiones pendientes:", sum(1 for v in MI_CLASIFICACION.values() if v is None))

### Pruebas de concepto: requisitos 1 y 4

Diez corridas de una función determinista y diez de GPT-2 con el mismo texto. Se cuentan salidas distintas y latencia. Antes de ejecutar, escribe tu predicción.

In [ ]:
# TODO: ¿cuántas salidas distintas esperas de GPT-2 en 10 corridas? (un entero por prueba)
PREDICCION_POC = {"correo": None, "reembolso": None}

def correr_10(nombre, metodo, fn):
    salidas, tiempos = [], []
    for _ in range(10):
        t0 = time.perf_counter()
        s = fn()
        tiempos.append(time.perf_counter() - t0)
        salidas.append(s if isinstance(s, str) else repr(s))
    frecuente, veces = Counter(salidas).most_common(1)[0]
    return {"Requisito": nombre, "Método": metodo, "Salidas distintas (10)": len(set(salidas)),
            "Latencia media (s)": round(statistics.mean(tiempos), 4), "Salida más frecuente": frecuente[:60]}

TICKET_CORREO = "Escríbanme a ana.torres@example.com, necesito la factura de junio de nuevo."

def detectar_correo(texto):
    m = re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", texto)
    return m.group(0) if m else None

prompt_correo = f"Texto: {TICKET_CORREO}\nCorreo electrónico del cliente:"

filas = [
    correr_10("1. Detectar correo", "Función determinista", lambda: detectar_correo(TICKET_CORREO)),
    correr_10("1. Detectar correo", "GPT-2", lambda: generar(prompt_correo, do_sample=True, temperature=0.8, max_new_tokens=10)[0].strip()),
]
print("Predicción para 'correo':", PREDICCION_POC["correo"])
print(pd.DataFrame(filas).to_markdown(index=False))

In [ ]:
def calcular_reembolso(monto_mensual, veces_cobrado):
    return monto_mensual * (veces_cobrado - 1)

prompt_reembolso = "El cliente pagó 599 pesos dos veces por error. El reembolso que corresponde es de"

filas = [
    correr_10("4. Calcular reembolso", "Función determinista", lambda: calcular_reembolso(599, 2)),
    correr_10("4. Calcular reembolso", "GPT-2", lambda: generar(prompt_reembolso, do_sample=True, temperature=0.8, max_new_tokens=8)[0].strip()),
]
print("Predicción para 'reembolso':", PREDICCION_POC["reembolso"])
print(pd.DataFrame(filas).to_markdown(index=False))
print("\nRespuesta correcta: 599")

## Sección 2. Arquitectura V0 de la mesa de soporte

El diagrama se dibuja fuera del notebook (foto o imagen). Aquí se llena la tabla que acompaña al diagrama y las siete preguntas.

In [ ]:
# TODO: una decisión y una justificación por fila.
ARQUITECTURA_V0 = {
    "Entrada":            {"decision": "", "justificacion": ""},
    "Núcleo: prompt":     {"decision": "", "justificacion": ""},
    "Núcleo: modelo":     {"decision": "", "justificacion": ""},
    "Núcleo: validación": {"decision": "", "justificacion": ""},
    "Registro":           {"decision": "", "justificacion": ""},
    "Lo determinista":    {"decision": "", "justificacion": ""},
}

# TODO: responde las siete preguntas para el uso del LLM en la mesa de soporte.
SIETE_PREGUNTAS = {
    "¿Qué problema resuelve?": "",
    "¿Cómo funciona?": "",
    "¿Cuándo usarlo?": "",
    "¿Cuándo no usarlo?": "",
    "¿Cómo evaluarlo?": "",
    "¿Qué puede fallar?": "",
    "¿Cuánto cuesta?": "",
}

print(pd.DataFrame([{"Fila": k, **v} for k, v in ARQUITECTURA_V0.items()]).to_markdown(index=False))
print()
for i, (p, r) in enumerate(SIETE_PREGUNTAS.items(), start=1):
    print(f"{i}. {p} {r}")
print("\nPendientes:", sum(1 for v in ARQUITECTURA_V0.values() if not v["decision"]) + sum(1 for r in SIETE_PREGUNTAS.values() if not r), "campos vacíos")

## Sección 3. Validación y registro

GPT-2 clasifica un ticket con un prompt de tres ejemplos. Un validador determinista decide si la salida sirve y un registro guarda cada llamada. Se corre 20 veces con muestreo (temperature 0.7) para observar varianza.

El ticket y las categorías van en inglés porque GPT-2 se entrenó casi solo con inglés (la semana 3 explica por qué). Con el modelo de la semana 4 todo vuelve al español.

In [ ]:
OPCIONES = {"billing", "access", "bug"}

TICKET = "I cannot log in, it says my password is wrong even though I just changed it."
ESPERADA = "access"

def construir_prompt(texto):
    return ('Ticket: "I need a refund for the duplicate charge."\nCategory: billing\n\n'
            'Ticket: "The login page returns an error."\nCategory: access\n\n'
            'Ticket: "The export button does nothing in Safari."\nCategory: bug\n\n'
            f'Ticket: "{texto}"\nCategory:')

def validar(salida):
    """v1: primera línea, sin espacios a los lados, pertenencia exacta al conjunto."""
    candidato = salida.split("\n")[0].strip()
    return candidato if candidato in OPCIONES else None

def clasificar(texto, validador, registro):
    prompt = construir_prompt(texto)
    salida, dt = generar(prompt, do_sample=True, temperature=0.7, max_new_tokens=4)
    etiqueta = validador(salida)
    registro.append({"salida_cruda": salida, "etiqueta": etiqueta, "valida": etiqueta is not None,
                     "correcta": etiqueta == ESPERADA, "latencia_s": round(dt, 3)})
    return etiqueta

def correr_20(validador):
    torch.manual_seed(0)  # misma secuencia de sorteos para comparar validadores
    registro = []
    for _ in range(20):
        clasificar(TICKET, validador, registro)
    df = pd.DataFrame(registro)
    resumen = {"pasan": int(df["valida"].sum()), "correctas_entre_las_que_pasan": int(df["correcta"].sum()),
               "fallas_formato": int((~df["valida"]).sum())}
    return df, resumen

registro_v1, resumen_v1 = correr_20(validar)
print("v1:", resumen_v1)
print(registro_v1[["salida_cruda", "etiqueta", "valida", "correcta"]].to_markdown(index=False))

In [ ]:
def validar_v2(salida):
    """TODO: normaliza la salida antes de comprobar pertenencia.
    Ideas: minúsculas, quitar signos de puntuación, tomar solo la primera palabra.
    Cuidado con buscar la opción como subcadena: "debug" contiene "bug".
    Pregunta para el reporte: "login" y "error" son plausibles y no están en el conjunto.
    ¿Mapearlos a "access" y "bug" es validar o es decidir? Si lo haces, justifícalo.
    Mientras no lo escribas, se comporta igual que v1."""
    return validar(salida)

registro_v2, resumen_v2 = correr_20(validar_v2)
comparacion = pd.DataFrame([{"Validador": "validar (v1)", **resumen_v1}, {"Validador": "validar_v2", **resumen_v2}])
print(comparacion.to_markdown(index=False))
print("\nSalidas crudas que v1 rechazó, con su frecuencia (para decidir qué normalizar):")
for s, n in Counter(registro_v1.loc[~registro_v1["valida"], "salida_cruda"]).most_common(10):
    print(n, repr(s))

In [ ]:
# TODO: política de falla cuando la validación no pasa: "reintentar", "degradar" (categoría por defecto) o "humano".
POLITICA_DE_FALLA = ""
JUSTIFICACION_POLITICA = ""

print("Política:", POLITICA_DE_FALLA or "(pendiente)")
print("Justificación:", JUSTIFICACION_POLITICA or "(pendiente)")

## Resumen para el reporte

Esta celda junta los números que el reporte pide.

In [ ]:
print("Predicciones PoC:", PREDICCION_POC)
print("Validación v1:", resumen_v1)
print("Validación v2:", resumen_v2)
print("Decisiones de requisitos llenas:", sum(1 for v in MI_CLASIFICACION.values() if v), "de 10")
print("Latencia media de GPT-2 en las 20 corridas (s):", round(registro_v1["latencia_s"].mean(), 3))